In [1]:
import os
import xarray as xr

In [2]:
# === Path Builder ===
def get_file_paths(scenario, ens_num):
    num = f"{ens_num:02d}"
    files = []

    if scenario == "ARISE":
        end = "206912" if ens_num not in [8, 9] else "207012"
        path = os.path.join(
            "/glade/campaign/cesm/collections/ARISE-SAI-1.5/",
            f"b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.0{num}/atm/proc/tseries/month_1/",
            f"b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.0{num}.cam.h0.O3.203501-{end}.nc"
        )
        files.append(path)

    elif scenario == "SSP245":
        base = os.path.join(
            "/glade/campaign/cesm/collections/CESM2-WACCM-SSP245/",
            f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{num}/atm/proc/tseries/month_1/"
        )
        files.append(os.path.join(base, f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{num}.cam.h0.O3.201501-206412.nc"))
        end = "210012" if ens_num <= 5 else "206912"
        files.append(os.path.join(base, f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{num}.cam.h0.O3.206501-{end}.nc"))

    return files

In [3]:
# === Processing Function ===
def fix_months(da, expected_start, expected_end):
    """Fix time and crop"""
    expected_dates = xr.date_range(
        start=expected_start,
        end=expected_end,
        freq="MS",
        calendar="noleap",
        use_cftime=True
    )

    if len(da.time) != len(expected_dates):
        raise ValueError("Time dimension length mismatch with expected range.")

    da["time"] = expected_dates

    # Crop to desired range
    if scenario == "ARISE":
        start_date = "2035-01"
    else:
        start_date = "2020-01"
    da = da.sel(time=slice(start_date, "2069-12"))
    return da

In [4]:
# === Update based on your environment ===
SAVE_DIR = "/glade/work/awells/air_quality/CESM/O3/"
SCENARIOS = ["ARISE", "SSP245"]


# === Main Loop ===
for scenario in SCENARIOS:
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, ensemble {ens_num:02d}")
        file_list = get_file_paths(scenario, ens_num)
        datasets = []

        for f in file_list:
            if not os.path.exists(f):
                raise ValueError(f"Missing: {f}")

            print(f"Reading {os.path.basename(f)}")
            datasets.append(xr.open_dataset(f)["O3"])

        # Combine files if multiple
        combined_ds = xr.concat(datasets, dim="time") if len(datasets) > 1 else datasets[0]

        if scenario == "ARISE":
            try:
                expected_end = "2070-12" if ens_num in [8, 9] else "2069-12"
                annual_pm25 = fix_months(combined_ds, "2035-01", expected_end)
            except Exception as e:
                print(f"Error processing ensemble {ens_num:02d}: {e}")
                continue

        if scenario == "SSP245":
            try:
                expected_end = "2100-12" if ens_num <= 5 else "2069-12"
                annual_pm25 = fix_months(combined_ds, "2015-01", expected_end)
            except Exception as e:
                print(f"Error processing ensemble {ens_num:02d}: {e}")
                continue

        # Save output
        if scenario == "ARISE":
            dates = "203501-206912"
        else:
            dates = "202001-206912"
        out_file = f"O3_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving monthly O3 to {out_path}")
        annual_pm25.to_netcdf(out_path)

print("Done processing all O3 ensembles.")

Processing SSP245, ensemble 10
Reading b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.010.cam.h0.O3.201501-206412.nc
Reading b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.010.cam.h0.O3.206501-206912.nc
Saving monthly O3 to /glade/work/awells/air_quality/CESM/O3/O3_CESM2_SSP245_10_202001-206912.nc
Done processing all O3 ensembles.
